# 👨‍🍳 Personal Chef Agent

Run every cell once, top to bottom. The last cell shows a small GUI where you can:

- Upload a photo of your fridge/pantry **or** type your ingredients
- Chat back and forth with the chef agent (it remembers the conversation and can search the web for recipes)

No need to re-run cells after that — everything happens inside the GUI.

## 1. Install dependencies

In [ ]:
!pip install -q langchain langchain-groq langgraph tavily-python ipywidgets

## 2. API keys

Works in Colab (uses your saved secrets if present) or in a plain Jupyter install (falls back to a hidden password prompt).

In [ ]:
import os
from getpass import getpass


def get_key(name: str, secret_name: str) -> str:
    # Try Colab secrets first, if available
    try:
        from google.colab import userdata
        value = userdata.get(secret_name)
        if value:
            return value
    except Exception:
        pass
    # Fall back to an existing environment variable
    value = os.environ.get(name)
    if value:
        return value
    # Otherwise, prompt for it once
    value = getpass(f"Enter your {name}: ")
    os.environ[name] = value
    return value


GROQ_API_KEY = get_key("GROQ_API_KEY", "groq_key")
TAVILY_API_KEY = get_key("TAVILY_API_KEY", "tavily_key")
print("API keys loaded.")

## 3. Tools, models, and agents

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver
from tavily import TavilyClient

tavily_client = TavilyClient(TAVILY_API_KEY)


@tool
def web_search(query: str) -> str:
    """Searches the web for the given input query."""
    response = tavily_client.search(query)
    if response.get("answer"):
        return response["answer"]
    elif "results" in response:
        return " ".join(
            r["content"] for r in response["results"] if "content" in r
        )[:3000]
    else:
        return "No relevant information found."


# Text model - used for ingredient cleanup and the chef conversation
llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=GROQ_API_KEY, temperature=0.1)

# Vision model - used only to read ingredients out of a photo
llm_image = ChatGroq(model="qwen/qwen3.6-27b", api_key=GROQ_API_KEY, temperature=0.1)

image_agent = create_agent(
    model=llm_image,
    system_prompt=(
        "Identify all food items and ingredients visible in the image. "
        "Your response MUST be a comma-separated list of these items, and "
        "ONLY these items. Do NOT include any conversational text, "
        "introductory phrases, explanations, or any form of thought "
        "process. Strictly adhere to outputting just the comma-separated "
        "list."
    ),
)

# Some vision models leak their reasoning ("<think>...</think>") even when
# told not to, so this cleanup agent normalizes whatever comes back into a
# strictly comma-separated ingredient list.
ingredients_agent = create_agent(
    model=llm,
    system_prompt=(
        "Extract and output only a comma separated list of ingredients "
        "from the provided text. Do not include any other commentary, "
        "reasoning, or formatting."
    ),
)

CHEF_SYSTEM_PROMPT = """
You are a friendly, knowledgeable personal chef.

The user will tell you what ingredients they have on hand. Using the
web_search tool, look up recipes that make good use of those ingredients
(prioritizing ones that use as many of them as possible). Offer a short list
of recipe suggestions, and if the user asks for full instructions on one of
them, search for and provide clear step-by-step instructions.

Keep responses conversational and concise. Ask a clarifying question if the
user's request is ambiguous (e.g. dietary restrictions, cuisine preference,
time available).
"""

chef_agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=CHEF_SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)

print("Agents ready.")

## 4. Helper functions

In [ ]:
import base64
import uuid


def extract_ingredients_from_image(img_bytes: bytes, mime_type: str = "image/png") -> str:
    """Runs the vision agent + cleanup agent on image bytes and returns a
    tidy comma-separated ingredient string."""
    img_b64 = base64.b64encode(img_bytes).decode("utf-8")

    multimodal_question = HumanMessage(
        content=[
            {
                "type": "text",
                "text": (
                    "Identify all food items and ingredients visible in the "
                    "image. Your response MUST be a comma-separated list of "
                    "these items, and ONLY these items. Do NOT include any "
                    "conversational text, introductory phrases, "
                    "explanations, or any form of thought process. Strictly "
                    "adhere to outputting just the comma-separated list. "
                    "For example: apple, banana, orange."
                ),
            },
            {"type": "image", "base64": img_b64, "mime_type": mime_type},
        ]
    )

    raw_response = image_agent.invoke({"messages": [multimodal_question]})
    raw_text = raw_response["messages"][-1].content

    cleaned = ingredients_agent.invoke({"messages": [HumanMessage(content=raw_text)]})
    return cleaned["messages"][-1].content.strip()


def ask_chef(message: str, thread_id: str) -> str:
    """Sends a message to the chef agent under a given conversation thread
    and returns its reply. The checkpointer keeps prior turns in memory, so
    repeated calls with the same thread_id continue the same conversation."""
    config = {"configurable": {"thread_id": thread_id}}
    response = chef_agent.invoke({"messages": [HumanMessage(content=message)]}, config)
    return response["messages"][-1].content

## 5. GUI

Upload a photo (optional) or type ingredients directly, click **Start chat**, then keep chatting in the box at the bottom.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

# --- state -----------------------------------------------------------
thread_id = str(uuid.uuid4())
chat_started = False

# --- widgets -----------------------------------------------------------
upload = widgets.FileUpload(accept="image/*", multiple=False, description="Upload photo")
scan_button = widgets.Button(description="Scan photo for ingredients", button_style="primary",
                              layout=widgets.Layout(width="220px"))

ingredients_box = widgets.Textarea(
    placeholder="...or just type your ingredients here, comma separated",
    layout=widgets.Layout(width="520px", height="60px"),
)
start_button = widgets.Button(description="Start chat", button_style="success",
                               layout=widgets.Layout(width="150px"))

status_label = widgets.HTML(value="")

chat_output = widgets.Output(
    layout=widgets.Layout(border="1px solid #ddd", height="320px", overflow_y="auto", padding="10px")
)

chat_input = widgets.Text(placeholder="Type a message and press Enter or click Send...",
                           layout=widgets.Layout(width="420px"))
send_button = widgets.Button(description="Send", button_style="info", layout=widgets.Layout(width="90px"))
chat_row = widgets.HBox([chat_input, send_button])
chat_row.layout.display = "none"  # hidden until chat starts


def add_bubble(sender: str, text: str):
    if sender == "user":
        html = f"""<div style="text-align:right;margin:6px 0;">
          <span style="background:#2f6fed;color:white;padding:8px 12px;border-radius:14px;
          display:inline-block;max-width:80%;text-align:left;">{text}</span></div>"""
    else:
        html = f"""<div style="text-align:left;margin:6px 0;">
          <span style="background:#f0f0f0;color:#222;padding:8px 12px;border-radius:14px;
          display:inline-block;max-width:80%;">🍳 {text}</span></div>"""
    with chat_output:
        display(HTML(html))


def on_scan_clicked(b):
    if not upload.value:
        status_label.value = "<i style='color:#b00'>Please upload an image first.</i>"
        return
    status_label.value = "<i>Reading your photo...</i>"

    # ipywidgets v7 vs v8 have different .value shapes
    file_info = upload.value[0] if isinstance(upload.value, tuple) else list(upload.value.values())[0]
    content = file_info["content"]
    img_bytes = bytes(content)
    name = file_info.get("name", "photo.png") if isinstance(file_info, dict) else "photo.png"
    ext = name.rsplit(".", 1)[-1].lower() if "." in name else "png"
    mime_type = f"image/{'jpeg' if ext == 'jpg' else ext}"

    try:
        ingredients = extract_ingredients_from_image(img_bytes, mime_type)
        ingredients_box.value = ingredients
        status_label.value = "<i style='color:#0a0'>Ingredients extracted below — edit if needed, then click Start chat.</i>"
    except Exception as e:
        status_label.value = f"<i style='color:#b00'>Couldn't read that image: {e}</i>"


def on_start_clicked(b):
    global chat_started
    ingredients = ingredients_box.value.strip()
    if not ingredients:
        status_label.value = "<i style='color:#b00'>Add some ingredients first (photo or text).</i>"
        return

    start_button.disabled = True
    scan_button.disabled = True
    status_label.value = "<i>Chef is thinking...</i>"

    message = f"I have the following ingredients: {ingredients}. What can I make?"
    add_bubble("user", message)
    reply = ask_chef(message, thread_id)
    add_bubble("chef", reply)

    status_label.value = ""
    chat_row.layout.display = "flex"
    chat_started = True


def handle_send(_=None):
    if not chat_started:
        return
    message = chat_input.value.strip()
    if not message:
        return
    chat_input.value = ""
    add_bubble("user", message)
    status_label.value = "<i>Chef is thinking...</i>"
    reply = ask_chef(message, thread_id)
    status_label.value = ""
    add_bubble("chef", reply)


scan_button.on_click(on_scan_clicked)
start_button.on_click(on_start_clicked)
send_button.on_click(handle_send)
chat_input.on_submit(handle_send)  # pressing Enter also sends

display(widgets.VBox([
    widgets.HTML("<h3>🥕 What have you got?</h3>"),
    widgets.HBox([upload, scan_button]),
    ingredients_box,
    start_button,
    status_label,
    widgets.HTML("<hr>"),
    chat_output,
    chat_row,
]))